In [1]:
from ollama import chat

In [2]:
def get_discount(customer_type: str) -> int:
    """
    Müşteri türüne göre indirim oranını döndürür.

    Args:
        customer_type: Müşterinin türü.
    """
    discounts = {
        "standard": 0,
        "silver": 5,
        "gold": 10
    }

    return discounts.get(customer_type.lower(), 0)

In [3]:
def calculate_price(
    price: float,
    discount_rate: float
) -> str:
    """
    İndirimli ürün fiyatını hesaplar.

    Args:
        price: Ürünün normal fiyatı.
        discount_rate: İndirim oranı.
    """
    final_price = price * (1 - discount_rate / 100)

    return f"{final_price:.2f} TL"

In [4]:
available_tools = {
    "get_discount": get_discount,
    "calculate_price": calculate_price
}

In [5]:
messages = [
    {
        "role": "system",
        "content": """
Sen bir alışveriş agent'ısın.

İndirim oranlarını tahmin etme.
Önce get_discount tool'unu kullan.

Fiyat hesaplamalarını kendin yapma.
calculate_price tool'unu kullan.

Gerekli bütün sonuçlar geldikten sonra
kullanıcıya kısa bir final cevap ver.
"""
    },
    {
        "role": "user",
        "content": """
Gold müşterimiz 850 TL'lik bir ürün alıyor.
İndirimli fiyatı hesapla.
"""
    }
]

In [7]:
from ollama import chat

# 1. Araçların (Tools) Tanımlanması
def get_discount(customer_type: str) -> int:
    """
    Müşteri türüne göre indirim oranını döndürür.

    Args:
        customer_type: Müşterinin türü.
    """
    discounts = {
        "standard": 0,
        "silver": 5,
        "gold": 10
    }
    return discounts.get(customer_type.lower(), 0)


def calculate_price(price: float, discount_rate: float) -> str:
    """
    İndirimli ürün fiyatını hesaplar.

    Args:
        price: Ürünün normal fiyatı.
        discount_rate: İndirim oranı.
    """
    final_price = price * (1 - discount_rate / 100)
    return f"{final_price:.2f} TL"


# 2. Araç Haritasının ve Mesaj Geçmişinin Hazırlanması
available_tools = {
    "get_discount": get_discount,
    "calculate_price": calculate_price
}

messages = [
    {
        "role": "system",
        "content": """
Sen bir alışveriş agent'ısın.

İndirim oranlarını tahmin etme.
Önce get_discount tool'unu kullan.

Fiyat hesaplamalarını kendin yapma.
calculate_price tool'unu kullan.

Gerekli bütün sonuçlar geldikten sonra
kullanıcıya kısa bir final cevap ver.
"""
    },
    {
        "role": "user",
        "content": """
Gold müşterimiz 850 TL'lik bir ürün alıyor.
İndirimli fiyatı hesapla.
"""
    }
]

# 3. ReAct Ajan Döngüsü (Thought -> Action -> Observation)
for step in range(5):
    response = chat(
        model="qwen3.5:9b",
        messages=messages,
        tools=list(available_tools.values()),
        think=True,
        stream=False
    )

    # Modelin ürettiği yanıtı mesaj geçmişine ekle
    messages.append(response.message)

    # Eğer model herhangi bir araç çağırmadıysa nihai cevabı vermiştir
    if not response.message.tool_calls:
        print("\nFINAL CEVAP:")
        print(response.message.content)
        break

    # Model araç/araçlar çağırdıysa eylemleri (Action) çalıştır
    for tool_call in response.message.tool_calls:
        tool_name = tool_call.function.name
        tool_args = tool_call.function.arguments

        print(f"\nADIM {step + 1}")
        print("ACTION:", tool_name)
        print("ARGÜMANLAR:", tool_args)

        tool_function = available_tools.get(tool_name)

        if tool_function:
            result = tool_function(**tool_args)
        else:
            result = f"Hata: {tool_name} isimli tool bulunamadı."

        print("OBSERVATION:", result)

        # Araç çalıştırma sonucunu (Gözlem) mesaj geçmişine kaydet
        messages.append(
            {
                "role": "tool",
                "tool_name": tool_name,
                "content": str(result)
            }
        )
else:
    # 5 adım içinde break ile çıkılamadıysa çalışır
    print("Maksimum adım sayısına ulaşıldı.")

# 4. (Opsiyonel) Tüm Mesaj Geçmişini Yazdırma
print("\nMESAJ GEÇMİŞİ:")
for message in messages:
    if isinstance(message, dict):
        print(message["role"], "->", message.get("content", ""))
    else:
        print(message.role, "->", message.content or "Tool çağrısı")


ADIM 1
ACTION: get_discount
ARGÜMANLAR: {'customer_type': 'Gold'}
OBSERVATION: 10

ADIM 2
ACTION: calculate_price
ARGÜMANLAR: {'price': 850, 'discount_rate': 10}
OBSERVATION: 765.00 TL

FINAL CEVAP:
Gold müşterimiz için indirim oranı %10'dur. Bu oranla 850 TL'lik ürünün indirimli fiyatı **765.00 TL** olarak hesaplandı.

MESAJ GEÇMİŞİ:
system -> 
Sen bir alışveriş agent'ısın.

İndirim oranlarını tahmin etme.
Önce get_discount tool'unu kullan.

Fiyat hesaplamalarını kendin yapma.
calculate_price tool'unu kullan.

Gerekli bütün sonuçlar geldikten sonra
kullanıcıya kısa bir final cevap ver.

user -> 
Gold müşterimiz 850 TL'lik bir ürün alıyor.
İndirimli fiyatı hesapla.

assistant -> Tool çağrısı
tool -> 10
assistant -> Tool çağrısı
tool -> 765.00 TL
assistant -> Gold müşterimiz için indirim oranı %10'dur. Bu oranla 850 TL'lik ürünün indirimli fiyatı **765.00 TL** olarak hesaplandı.
